# Performance Optimization & Vectorization in Pandas

## Definition
Writing **fast pandas code** means avoiding Python loops and using built-in vectorized operations.

### Performance Hierarchy (fastest → slowest)
1. **NumPy vectorized ops** on `.values` — fastest
2. **pandas vectorized ops** (arithmetic, `.str`, `.dt`) — very fast
3. **cython-optimized** methods (`.groupby`, `.rolling`) — fast
4. **`df.apply()` with axis=1** — ~100x slower than vectorized
5. **Python `for` loops** — ~1000x slower than vectorized

### Optimization Checklist
| Tip | Description |
|-----|-------------|
| Avoid `.apply(axis=1)` | Use vectorized math/np.where instead |
| Use `.values` or `.to_numpy()` | Drops pandas overhead for pure NumPy ops |
| Use `query()` | Often faster than boolean mask for large data |
| Use appropriate dtypes | float32 vs float64, category vs object |
| Use `inplace=True` carefully | Sometimes faster, often not needed |
| Profile first | `%timeit`, `line_profiler` |

In [ ]:
import pandas as pd
import numpy as np
import time

rng = np.random.default_rng(42)
N = 500_000
df = pd.DataFrame({
    'a': rng.uniform(0, 100, N),
    'b': rng.uniform(0, 100, N),
    'c': rng.integers(0, 10, N),
    'cat': rng.choice(['X', 'Y', 'Z'], N)
})
print('DataFrame shape:', df.shape)

def time_it(func, *args, label='', n=3):
    times = []
    for _ in range(n):
        t0 = time.time()
        func(*args)
        times.append(time.time() - t0)
    avg = sum(times) / n
    print(f'{label:45s}: {avg*1000:.2f}ms')

In [ ]:
# Comparison 1: Python loop vs vectorized arithmetic
def loop_add(df):
    result = []
    for i in range(len(df)):
        result.append(df['a'].iloc[i] + df['b'].iloc[i])
    return result

def vec_add(df):
    return df['a'] + df['b']

def numpy_add(df):
    return df['a'].values + df['b'].values

print('Addition benchmark:')
time_it(vec_add, df,   label='vectorized (pandas)')
time_it(numpy_add, df, label='numpy .values')
# loop too slow for 500k — skip or use small N
df_small = df.head(10000)
time_it(loop_add, df_small, label='python loop (10k only)')

In [ ]:
# Comparison 2: apply vs vectorized conditional
def apply_sigmoid(df):
    return df['a'].apply(lambda x: 1 / (1 + np.exp(-x/50)))

def vec_sigmoid(df):
    return 1 / (1 + np.exp(-df['a'].values / 50))

print('Sigmoid benchmark:')
time_it(vec_sigmoid, df, label='vectorized sigmoid')
time_it(apply_sigmoid, df, label='apply sigmoid')

In [ ]:
# Comparison 3: apply row-wise vs np.where
def apply_label(df):
    return df.apply(lambda row: 'High' if row['a'] + row['b'] > 100 else 'Low', axis=1)

def vec_label(df):
    return np.where(df['a'].values + df['b'].values > 100, 'High', 'Low')

print('Row labeling benchmark:')
time_it(vec_label, df, label='vectorized np.where')
time_it(apply_label, df.head(50000), label='apply row-wise (50k only)')

In [ ]:
# Comparison 4: Python loop groupby vs pandas groupby
def pandas_groupby_mean(df):
    return df.groupby('cat')['a'].mean()

def numpy_groupby(df):
    result = {}
    for cat in df['cat'].unique():
        result[cat] = df.loc[df['cat'] == cat, 'a'].values.mean()
    return result

print('Groupby benchmark:')
time_it(pandas_groupby_mean, df, label='pandas groupby().mean()')
time_it(numpy_groupby, df, label='manual numpy groupby')

In [ ]:
# Using .values for pure NumPy operations
arr_a = df['a'].values   # NumPy array, no pandas overhead
arr_b = df['b'].values

# Compute without pandas overhead
t0 = time.time()
result = np.sqrt(arr_a**2 + arr_b**2)
print(f'Pure NumPy Euclidean dist: {(time.time()-t0)*1000:.2f}ms')

t0 = time.time()
result2 = np.sqrt(df['a']**2 + df['b']**2)
print(f'pandas arithmetic:          {(time.time()-t0)*1000:.2f}ms')

In [ ]:
# query() vs boolean mask for large DataFrame
t0 = time.time()
r1 = df[df['a'] > 50]
print(f'Boolean mask: {(time.time()-t0)*1000:.2f}ms  rows={len(r1)}')

t0 = time.time()
r2 = df.query('a > 50')
print(f'query():      {(time.time()-t0)*1000:.2f}ms  rows={len(r2)}')

In [ ]:
# eval() — vectorized expression evaluation
t0 = time.time()
r1 = df['a'] * 2 + df['b'] ** 2 - df['a'] * df['b']
print(f'Standard arithmetic: {(time.time()-t0)*1000:.2f}ms')

t0 = time.time()
r2 = pd.eval('df.a * 2 + df.b ** 2 - df.a * df.b')
print(f'pd.eval():           {(time.time()-t0)*1000:.2f}ms')
print('Results match:', np.allclose(r1, r2))